# Configuração Inicial

In [1]:

import pandas as pd
import numpy as np
import warnings
import os

# Ignorar avisos futuros para uma saída mais limpa
warnings.filterwarnings('ignore')

print("Bibliotecas importadas com sucesso.")

Bibliotecas importadas com sucesso.


# Definição das Fontes de Dados

In [2]:

RAW_DIR = os.path.join("..", "data", "raw")

caminhos_arquivos = {
    "mrr.csv": os.path.join(RAW_DIR, "mrr.csv"),
    "clientes_desde.csv": os.path.join(RAW_DIR, "clientes_desde.csv"),
    "contratacoes_ultimos_12_meses.csv": os.path.join(RAW_DIR, "contratacoes_ultimos_12_meses.csv"),
    "nps_relacional.csv": os.path.join(RAW_DIR, "nps_relacional.csv")
}

# Carregar os dataframes em um dicionário
datasets = {}
for nome, caminho in caminhos_arquivos.items():
    try:
        # Usar 'sep=;' conforme identificado na sua EDA
        datasets[nome] = pd.read_csv(caminho, sep=';')
        print(f"Arquivo '{nome}' carregado com sucesso.")
    except FileNotFoundError:
        print(f"ERRO: Arquivo '{nome}' não encontrado em '{caminho}'.")

Arquivo 'mrr.csv' carregado com sucesso.
Arquivo 'clientes_desde.csv' carregado com sucesso.
Arquivo 'contratacoes_ultimos_12_meses.csv' carregado com sucesso.
Arquivo 'nps_relacional.csv' carregado com sucesso.


# Análise de Qualidade dos Dados (Data Quality Assessment)

In [3]:

def data_quality_report(df, nome_dataset):
    print(f"\n ANÁLISE DE QUALIDADE - {nome_dataset}")
    print("=" * 60)
    
    # Informações básicas
    print(f"\n INFORMAÇÕES BÁSICAS:")
    print(f"• Dimensões: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
    print(f"• Memória utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Tipos de dados
    print(f"\n TIPOS DE DADOS:")
    for col in df.columns:
        print(f"• {col}: {df[col].dtype}")
    
    # Valores ausentes
    print(f"\n VALORES AUSENTES:")
    missing_data = df.isnull().sum()
    missing_pct = (missing_data / len(df)) * 100
    
    for col in df.columns:
        if missing_data[col] > 0:
            print(f"• {col}: {missing_data[col]:,} ({missing_pct[col]:.1f}%)")
        else:
            print(f"• {col}:  Sem valores ausentes")
    
    # Duplicatas
    duplicates = df.duplicated().sum()
    print(f"\n DUPLICATAS:")
    print(f"• Total: {duplicates:,} ({duplicates/len(df)*100:.1f}%)")
    
    # Valores únicos
    print(f"\n VALORES ÚNICOS:")
    for col in df.columns:
        unique_count = df[col].nunique()
        print(f"• {col}: {unique_count:,} valores únicos")
    
    return {
        'missing_pct': missing_pct.mean(),
        'duplicates_pct': duplicates/len(df)*100,
        'memory_mb': df.memory_usage(deep=True).sum() / 1024**2
    }

quality_scores = {}
for nome, df in datasets.items():
    quality_scores[nome] = data_quality_report(df, nome)


 ANÁLISE DE QUALIDADE - mrr.csv

 INFORMAÇÕES BÁSICAS:
• Dimensões: 7,309 linhas × 2 colunas
• Memória utilizada: 0.44 MB

 TIPOS DE DADOS:
• CLIENTE: object
• MRR_12M: float64

 VALORES AUSENTES:
• CLIENTE:  Sem valores ausentes
• MRR_12M:  Sem valores ausentes

 DUPLICATAS:
• Total: 0 (0.0%)

 VALORES ÚNICOS:
• CLIENTE: 7,309 valores únicos
• MRR_12M: 7,151 valores únicos

 ANÁLISE DE QUALIDADE - clientes_desde.csv

 INFORMAÇÕES BÁSICAS:
• Dimensões: 10,615 linhas × 2 colunas
• Memória utilizada: 1.15 MB

 TIPOS DE DADOS:
• CLIENTE: object
• CLIENTE_DESDE: object

 VALORES AUSENTES:
• CLIENTE:  Sem valores ausentes
• CLIENTE_DESDE:  Sem valores ausentes

 DUPLICATAS:
• Total: 0 (0.0%)

 VALORES ÚNICOS:
• CLIENTE: 10,615 valores únicos
• CLIENTE_DESDE: 4,315 valores únicos

 ANÁLISE DE QUALIDADE - contratacoes_ultimos_12_meses.csv

 INFORMAÇÕES BÁSICAS:
• Dimensões: 4,314 linhas × 3 colunas
• Memória utilizada: 0.53 MB

 TIPOS DE DADOS:
• CD_CLIENTE: object
• QTD_CONTRATACOES_12M: in

# Conclusões sobre mrr.csv 

### Análise e Ações: `mrr.csv`

* **Diagnóstico:**
    * O dataset está limpo, sem valores nulos ou duplicatas.
    * A coluna `CLIENTE` é a chave de identificação.
    * A coluna `MRR_12M` já é do tipo `float64`, o que está correto.
* **Ação Corretiva no Pipeline:**
    1.  **Padronização:** Renomear a coluna `CLIENTE` para `id_cliente`.

# Conclusões sobre clientes_desde.csv

### Análise e Ações: `clientes_desde.csv`

* **Diagnóstico:**
    * Dataset limpo, sem nulos ou duplicatas.
    * A coluna `CLIENTE` é a chave de identificação.
    * A coluna `CLIENTE_DESDE` é do tipo `object` (texto) e precisa ser convertida.
* **Ação Corretiva no Pipeline:**
    1.  **Padronização:** Renomear a coluna `CLIENTE` para `id_cliente`.
    2.  **Tratamento de Tipos:** Converter a coluna `CLIENTE_DESDE` para o tipo `datetime`.

# Conclusões sobre contratacoes_ultimos_12_meses.csv

### Análise e Ações: `contratacoes_ultimos_12_meses.csv`

* **Diagnóstico:**
    * A coluna `VLR_CONTRATACOES_12M` é do tipo `object` devido ao uso da vírgula como separador decimal.
    * A chave de identificação é `CD_CLIENTE`.
* **Ação Corretiva no Pipeline:**
    1.  **Padronização:** Renomear a coluna `CD_CLIENTE` para `id_cliente`.
    2.  **Tratamento de Tipos:** Na coluna `VLR_CONTRATACOES_12M`, substituir a vírgula (`,`) por ponto (`.`) e converter para o tipo `float`.

# Conclusões sobre nps_relacional.csv

### Análise e Ações: `nps_relacional.csv`

* **Diagnóstico:**
    * Foram encontradas **201 linhas duplicadas (1.4%)**.
    * As colunas `Nota_*` apresentam uma **alta porcentagem de valores nulos** (de 43% a 58%).
    * A coluna de identificação do cliente é `metadata_codcliente`.
* **Ação Corretiva no Pipeline:**
    1.  **Padronização:** Renomear `metadata_codcliente` para `id_cliente`.
    2.  **Limpeza:** Remover as 201 linhas duplicadas.
    3.  **Tratamento de Nulos:** Implementar uma estratégia de preenchimento para os valores nulos nas colunas `Nota_*`. A **mediana** é uma escolha robusta, pois não é sensível a valores extremos.